In [1]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import numpy as np

from datetime import datetime
import time

In [2]:
current_time = datetime.now().strftime("%Y%m%d%H%M%S")

progress_file = f'/group/pmc021/amunif/epi-thesis/workflow/04. Neural Network/gpu_test_{current_time}.txt'
dataset_path = "/group/pmc021/amunif/epi-thesis/dataset/"

In [3]:
# Prepare the progress file
current_time = datetime.now().strftime("%Y%m%d%H%M%S")
progress_file = f'progress_conv1d_{current_time}.txt'

In [4]:
'''
User defined functions
'''
def load_large_csv(file_name, chunksize=20000):
    # Read the CSV file
    mylist = []

    for chunk in pd.read_csv(file_name, chunksize = chunksize):
        mylist.append(chunk)

    df = pd.concat(mylist, axis = 0)
    
    del mylist
    return df

def save_progress(file_name, message):
    with open(file_name, 'a+') as file:
        file.write(message + "\n")

In [5]:
X = pd.read_csv(f"{dataset_path}histone_features.csv", nrows=100)
X.head()

,h3k4me3_0,h3k4me3_1,h3k4me3_2,h3k4me3_3,h3k4me3_4,h3k4me3_5,h3k4me3_6,h3k4me3_7,h3k4me3_8,h3k4me3_9,...,h3k27me3_3990,h3k27me3_3991,h3k27me3_3992,h3k27me3_3993,h3k27me3_3994,h3k27me3_3995,h3k27me3_3996,h3k27me3_3997,h3k27me3_3998,h3k27me3_3999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
y = pd.read_csv(f"{dataset_path}value_1_df.csv", nrows=100)
y.head()

,h_value_1
0,0.000000
1,0.047392
2,0.000000
3,2.468570
4,2.468570


In [7]:
# Convert to numpy
X_np = X.to_numpy()
y_np = y.to_numpy()

In [8]:
# Split the data into training and testing
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

In [9]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
print(f'Using device: {device}')
save_progress(progress_file, f'Using device: {device}')

Using device: cpu


In [10]:
# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32).unsqueeze(1).to(device)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32).unsqueeze(1).to(device)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32).to(device)

In [11]:
# Create TensorDataset for training and testing sets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [12]:
# Create a DataLoader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [13]:
# Create CNN1D class
class HighDimCNN1D(nn.Module):
    def __init__(self, input_size):
        super(HighDimCNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        
        # Calculate the size after the convolution and pooling layers
        conv_output_size = input_size // 8  # Adjust this based on the number of pooling layers
        
        self.fc1 = nn.Linear(256 * conv_output_size, 512)
        self.fc2 = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.pool3(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [14]:
'''
Training
'''
input_size = 20000  # Number of features in the input
model = HighDimCNN1D(input_size=input_size).to(device)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
# Training loop
num_epochs = 100
for epoch in range(num_epochs): 
    start_time = time.time()
    
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    message = f'Epoch {epoch+1}, Time:{elapsed_time},  Loss: {loss.item()}'
    print(message)
    save_progress(progress_file, message)

Epoch 1, Time:9.900192737579346,  Loss: 72.07600402832031
Epoch 2, Time:9.016405582427979,  Loss: 94.02281188964844
Epoch 3, Time:8.963927507400513,  Loss: 114.66342163085938
Epoch 4, Time:9.036563873291016,  Loss: 36.40781021118164
Epoch 5, Time:9.209375858306885,  Loss: 142.20452880859375
Epoch 6, Time:9.886991739273071,  Loss: 154.9363555908203
Epoch 7, Time:9.759503602981567,  Loss: 99.1627426147461
Epoch 8, Time:9.310348749160767,  Loss: 4672.125
Epoch 9, Time:10.286764144897461,  Loss: 100.20706176757812
Epoch 10, Time:9.275325775146484,  Loss: 32.572593688964844
Epoch 11, Time:9.483944177627563,  Loss: 73.63834381103516
Epoch 12, Time:9.33066439628601,  Loss: 82.30669403076172
Epoch 13, Time:9.116858720779419,  Loss: 101.18565368652344
Epoch 14, Time:9.423644304275513,  Loss: 60.01161575317383
Epoch 15, Time:8.730194807052612,  Loss: 20.763944625854492
Epoch 16, Time:8.358730792999268,  Loss: 16.92530632019043
Epoch 17, Time:8.262501001358032,  Loss: 4.232501029968262
Epoch 18, 

In [16]:
'''
Evaluation
'''
# Evaluation
model.eval()
total_loss = 0.0
all_targets = []
all_predictions = []

# Evaluate the model
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        total_loss += loss.item()

        all_targets.extend(targets.cpu().numpy())
        all_predictions.extend(outputs.cpu().numpy())

print(f'Average loss on the test data: {total_loss/len(test_loader):.4f}')
save_progress(progress_file, f'Average loss on the test data: {total_loss/len(test_loader):.4f}')


Average loss on the test data: 2.7366


In [17]:
all_targets = np.array(all_targets)
all_predictions = np.array(all_predictions)

mse = mean_squared_error(all_targets, all_predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(all_targets, all_predictions)
r2 = r2_score(all_targets, all_predictions)

print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')
print(f'R2 Score: {r2}')

save_progress(progress_file, f'MSE: {mse}')
save_progress(progress_file, f'RMSE: {rmse}')
save_progress(progress_file, f'MAE: {mae}')
save_progress(progress_file, f'R2 Score: {r2}')

MSE: 3.0826332569122314
RMSE: 1.7557429075241089
MAE: 1.3563512563705444
R2 Score: 0.9982960820198059


In [18]:
# Create a DataFrame with true values and predictions
results_df = pd.DataFrame({
    'y_test': all_targets.flatten(),
    'y_pred': all_predictions.flatten()
})

# Save the DataFrame to a CSV file
results_df.to_csv(f'predictions_conv1d_{current_time}.csv', index=False)

print(f"Predictions and true values saved to predictions_conv1d_{current_time}.csv.")

Predictions and true values saved to predictions_conv1d_20240704075429.csv.
